In [39]:
import pandas as pd
import numpy as np
jobs=pd.read_csv('data/fake_job_postings.csv')

In [40]:
from codecarbon import EmissionsTracker

In [41]:
import platform
os_name = platform.system()

**PREPROCESSING**

In [42]:
jobs['fraudulent'].value_counts()

fraudulent
0    17014
1      866
Name: count, dtype: int64

In [43]:
indices_to_drop = jobs[jobs['fraudulent'] == 0].sample(n=16000).index #fixing class imbalance
jobs = jobs.drop(indices_to_drop)

In [44]:
jobs['fraudulent'].value_counts()

fraudulent
0    1014
1     866
Name: count, dtype: int64

**Commitments made in the project plan**

1. Methods: Logistic Regression, SVM, RNN, Random Forest, BERT Transformer
2. Utilize the same or similar pre-processing technique
3. I will then train and test each model, evaluating
accuracy, precision, recall, and the macro-averaged F1 score and utilizing CodeCarbon’s ability to
estimate CO2 Emissions to log and obtain the emissions from training the data and report the calculated
CE_Rel and delta CE_rel metrics.

In [45]:
jobs.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
6,7,Head of Content (m/f),"DE, BE, Berlin",ANDROIDPIT,20000-28000,"Founded in 2009, the Fonpit AG rose with its i...",Your Responsibilities: Manage the English-spea...,Your Know-How: ...,Your Benefits: Being part of a fast-growing co...,0,1,1,Full-time,Mid-Senior level,Master's Degree,Online Media,Management,0
7,8,Lead Guest Service Specialist,"US, CA, San Francisco",NaN,NaN,Airenvy’s mission is to provide lucrative yet ...,Who is Airenvy?Hey there! We are seasoned entr...,"Experience with CRM software, live chat, and p...",Competitive Pay. You'll be able to eat steak e...,0,1,1,NaN,NaN,NaN,NaN,NaN,0
26,27,Marketing Exec,"SG, ,",Marketing,NaN,If working in a cubical seems like your idea o...,We are currently expanding our Marketing Depar...,"This position is Junior to Mid level. Ideally,...",We are looking for Singaporean residents or in...,0,1,0,Full-time,Associate,NaN,Online Media,Marketing,0


In [46]:
model_df = jobs[['description', 'fraudulent']]

In [67]:
model_df.head()


,description,fraudulent
18,Kettle is hiring a Visual Designer!Job Locatio...,0
27,HAAD/DHA Licensed Doctors Opening in UAEWe the...,0
32,Construction: Entry-Level Craftsman Associate ...,0
39,"The Receptionist will be based in Chicago, IL....",0
47,The Customer Service Associate will be based i...,0


In [47]:
import nltk
import re
import html
from nltk.corpus import stopwords
nltk.download('stopwords',quiet=True)
stop_words=set(stopwords.words('english'))


def preprocess_text(text):
  text=re.sub(r"(?:http\S+|@)","",text) #arguments are pattern,replace,string.
  text=html.unescape(text) #convert XML to string. This function can handle XML entities like &amp
  tokens=text.split() #just using split()
  tokens=[token for token in tokens if token not in stop_words]
  return ' '.join(tokens)

there's a class imbalance. much more non-fraudulent ones than fraudulent. There are ~17800 observations and maybe 800 fraudulent ones. Maybe consider fixing this (but don't think it has a direct effect on workload, but maybe want things to be realistic).

In [ ]:
##CHECK LAB 6 FOR K FOLD cross validation applied to logistic regression

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score

In [49]:
model_df['description']=model_df['description'].astype(str)

/var/folders/tl/0pf8lcrd691gwn8392x3hs6r0000gn/T/ipykernel_1076/3832600678.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df['description']=model_df['description'].astype(str)


In [50]:
processed_descriptions=[preprocess_text(doc) for doc in model_df['description']]

In [51]:
labels=model_df['fraudulent']

In [52]:
desclist = list(processed_descriptions)
tokenized = []
for doc in desclist:
    sent = doc.lower()
    sent = doc.split()
    tokenized.append(sent)


In [53]:
glove_dir = '../glove.6B.100d.txt'
glove_index = {}
with open(glove_dir, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        embeddings = np.asarray(values[1:], dtype='float32')
        glove_index[word]=embeddings

In [54]:
sentence_embedding_list = []
for sentence in tokenized:
    word_embeddings = [glove_index.get(word) if word in glove_index.keys() else np.zeros(100) for word in sentence]
    if np.sum(word_embeddings)==0:
        sentence_embeddings = np.zeros(100)
    else:
        sentence_embeddings = np.mean(word_embeddings, axis = 0)
    sentence_embedding_list.append(sentence_embeddings)

In [55]:
np.array(sentence_embedding_list) #this should work if the loop ran correctly

array([[-0.02687274,  0.10430751,  0.11098192, ..., -0.13213153,
         0.41003661,  0.0978855 ],
       [ 0.06703218,  0.07334333,  0.0626333 , ..., -0.32179971,
         0.29371182,  0.21877906],
       [-0.14333169,  0.03167676,  0.07422771, ..., -0.18521696,
         0.41544255,  0.27948233],
       ...,
       [-0.02229362,  0.10114832, -0.00494408, ..., -0.17157776,
         0.47225002,  0.22366059],
       [ 0.00858834,  0.0652518 ,  0.10977765, ..., -0.10479753,
         0.33746191,  0.14930124],
       [-0.08891675, -0.02450465,  0.08813506, ..., -0.14956958,
         0.34873727,  0.09657894]])

**LOGISTIC REGRESSION**

In [56]:
experiment_name = 'logistic_regression'
tracker = EmissionsTracker(
    output_dir='codecarbon_mac/',
    output_file=f'{os_name}_{experiment_name}_emissions.csv')

[codecarbon WARNING @ 18:27:32] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 18:27:32] Error while trying to count physical CPUs: [Errno 2] No such file or directory: 'lscpu'. Defaulting to 1.
[codecarbon INFO @ 18:27:32] [setup] RAM Tracking...
[codecarbon INFO @ 18:27:32] [setup] CPU Tracking...
[codecarbon INFO @ 18:27:32] Tracking Apple CPU and GPU via PowerMetrics
[codecarbon INFO @ 18:27:32] [setup] GPU Tracking...
[codecarbon INFO @ 18:27:32] No GPU found.
[codecarbon INFO @ 18:27:32] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: PowerMetrics
                GPU Tracking Method: PowerMetrics
            
[codecarbon INFO @ 18:27:32] >>> Tracker's metadata:
[codecarbon INFO @ 18:27:32]   Platform system: macOS-15.6-arm64-arm-64bit
[codecarbon INFO @ 18:27:32]   Python version: 3.10.20
[codecarbon INFO @ 18:27:32]   CodeCarbon version: 

In [57]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(sentence_embedding_list, labels, 
                                                    test_size = 0.3,
                                                    random_state=42)


In [58]:
from sklearn.model_selection import cross_validate, KFold, GridSearchCV
from sklearn.base import TransformerMixin
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

In [59]:
C = [1,10,1000,10000]
penalty = ['l2', None]

params = {
    'classify__C': C,
    'classify__penalty': penalty
}

In [60]:
pipe_LR = Pipeline([
    ('scale', StandardScaler()),
    ('classify', LogisticRegression())])

In [61]:
inner_cv = KFold(n_splits = 3, shuffle=True, random_state=1)
outer_cv = KFold(n_splits = 5, shuffle = True, random_state=1)

grid_LR = GridSearchCV(pipe_LR, params, cv=inner_cv)

In [62]:
tracker.start()

In [63]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
scores = cross_validate(grid_LR,
                        X=X_train,
                        y=y_train,
                        cv=outer_cv,
                        scoring = ['accuracy', 'precision','recall','f1'],
                        return_estimator=True)

[codecarbon INFO @ 18:28:10] Energy consumed for RAM : 0.000013 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:28:11] Energy consumed for all CPUs : 0.000000 kWh. Total CPU Power : 0.0368 W
[codecarbon INFO @ 18:28:13] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0027000000000000006 W
[codecarbon INFO @ 18:28:13] 0.000013 kWh of electricity and 0.000000 L of water were used since the beginning.


In [64]:
print(scores['test_accuracy'])
print(scores['test_precision'])
print(scores['test_recall'])
print(scores['test_f1'])

[0.72727273 0.74144487 0.71863118 0.7338403  0.7148289 ]
[0.6984127  0.73043478 0.72519084 0.67479675 0.73148148]
[0.72131148 0.69421488 0.71428571 0.73451327 0.632     ]
[0.70967742 0.71186441 0.71969697 0.70338983 0.67811159]


In [65]:
grid_LR.fit(X_train,y_train)
grid_LR.best_params_

{'classify__C': 1, 'classify__penalty': 'l2'}

[codecarbon INFO @ 18:28:25] Energy consumed for RAM : 0.000023 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:28:26] Energy consumed for all CPUs : 0.000000 kWh. Total CPU Power : 0.046 W
[codecarbon INFO @ 18:28:28] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:28:28] 0.000023 kWh of electricity and 0.000000 L of water were used since the beginning.


In [19]:
#tracker.start()

In [66]:
reg_classifier=LogisticRegression(penalty='l2', C=1)
reg_classifier.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


[codecarbon INFO @ 18:28:40] Energy consumed for RAM : 0.000033 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:28:41] Energy consumed for all CPUs : 0.000001 kWh. Total CPU Power : 0.0729 W
[codecarbon INFO @ 18:28:43] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:28:43] 0.000033 kWh of electricity and 0.000000 L of water were used since the beginning.


In [67]:
y_pred = reg_classifier.predict(X_test) #should break to test train

In [68]:
print(classification_report(y_test, y_pred)) 

              precision    recall  f1-score   support

           0       0.77      0.78      0.77       312
           1       0.72      0.71      0.72       252

    accuracy                           0.75       564
   macro avg       0.75      0.74      0.74       564
weighted avg       0.75      0.75      0.75       564



[codecarbon INFO @ 18:28:55] Energy consumed for RAM : 0.000043 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:28:56] Energy consumed for all CPUs : 0.000001 kWh. Total CPU Power : 0.0248 W
[codecarbon INFO @ 18:28:58] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0027000000000000006 W
[codecarbon INFO @ 18:28:58] 0.000044 kWh of electricity and 0.000000 L of water were used since the beginning.


In [69]:
logistic_emissions = tracker.stop()

[codecarbon INFO @ 18:29:07] Energy consumed for RAM : 0.000051 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:29:09] Energy consumed for all CPUs : 0.000001 kWh. Total CPU Power : 0.024799999999999996 W
[codecarbon INFO @ 18:29:10] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:29:10] 0.000052 kWh of electricity and 0.000000 L of water were used since the beginning.


In [70]:
print(f'Emissions from this training run: {logistic_emissions:5f} kg CO2 eq')

Emissions from this training run: 0.000024 kg CO2 eq


**SUPPORT VECTOR MACHINE**

In [71]:
experiment_name = 'svm'
svmtracker = EmissionsTracker(
    output_dir='codecarbon_mac/',
    output_file=f'{os_name}_{experiment_name}_emissions.csv')

[codecarbon WARNING @ 18:32:04] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 18:32:04] Error while trying to count physical CPUs: [Errno 2] No such file or directory: 'lscpu'. Defaulting to 1.
[codecarbon INFO @ 18:32:04] [setup] RAM Tracking...
[codecarbon INFO @ 18:32:04] [setup] CPU Tracking...
[codecarbon INFO @ 18:32:04] Tracking Apple CPU and GPU via PowerMetrics
[codecarbon INFO @ 18:32:04] [setup] GPU Tracking...
[codecarbon INFO @ 18:32:04] No GPU found.
[codecarbon INFO @ 18:32:04] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: PowerMetrics
                GPU Tracking Method: PowerMetrics
            
[codecarbon INFO @ 18:32:04] >>> Tracker's metadata:
[codecarbon INFO @ 18:32:04]   Platform system: macOS-15.6-arm64-arm-64bit
[codecarbon INFO @ 18:32:04]   Python version: 3.10.20
[codecarbon INFO @ 18:32:04]   CodeCarbon version: 

In [72]:
class SparsetoDense(TransformerMixin):
  def fit(self, x, y = None, **fit_params):
    return self
  def transform(self, x, y=None, **fit_params):
    return x.toarray()

In [73]:
svm_pipe=Pipeline([
    ('scale', StandardScaler()),
    ('classify',SVC())
])

kernel= ['rbf', 'linear']
C = [0.001, 0.01, 1, 10] #if its running too long will cut down on some of these combos
svm_params = {
    'classify__kernel': kernel,
    'classify__C': C
}

In [74]:
inner_cv=KFold(n_splits=3, shuffle=True, random_state=1)
outer_cv=KFold(n_splits=5, shuffle=True, random_state=1)

grid_SVC=GridSearchCV(svm_pipe, svm_params, cv=inner_cv)

In [75]:
svmtracker.start()

In [76]:
scores=cross_validate(grid_SVC,
                     X=X_train,
                     y=y_train,
                     cv=outer_cv,
                     scoring=['accuracy','f1','precision','recall'],
                     return_estimator=True)

[codecarbon INFO @ 18:32:37] Energy consumed for RAM : 0.000013 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:32:38] Energy consumed for all CPUs : 0.000026 kWh. Total CPU Power : 6.3568 W
[codecarbon INFO @ 18:32:40] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:32:40] 0.000039 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 18:32:52] Energy consumed for RAM : 0.000023 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:32:53] Energy consumed for all CPUs : 0.000027 kWh. Total CPU Power : 0.030199999999999998 W
[codecarbon INFO @ 18:32:55] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0036 W
[codecarbon INFO @ 18:32:55] 0.000049 kWh of electricity and 0.000000 L of water were used since the beginning.


In [77]:
print(scores['test_accuracy'])
print(scores['test_precision'])
print(scores['test_recall'])
print(scores['test_f1'])

[0.8030303  0.82509506 0.81749049 0.87452471 0.82509506]
[0.7734375  0.79527559 0.85714286 0.82258065 0.83193277]
[0.81147541 0.83471074 0.76691729 0.90265487 0.792     ]
[0.792      0.81451613 0.80952381 0.86075949 0.81147541]


In [ ]:
#SVM Cross validation ran for 24 mins (wait it was faster locally on m4)

In [78]:
grid_SVC.fit(X_train,y_train)
grid_SVC.best_params_

{'classify__C': 10, 'classify__kernel': 'rbf'}

[codecarbon INFO @ 18:33:07] Energy consumed for RAM : 0.000033 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:33:08] Energy consumed for all CPUs : 0.000027 kWh. Total CPU Power : 0.0463 W
[codecarbon INFO @ 18:33:10] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0081 W
[codecarbon INFO @ 18:33:10] 0.000060 kWh of electricity and 0.000000 L of water were used since the beginning.


In [79]:
#bestmodel as SVC object
#then do bestmodel.fit
#then get y_pred with bestmodel
SVC_model = SVC(kernel='rbf', C=10)
SVC_model.fit(X_train, y_train)
y_pred = SVC_model.predict(X_test)

In [80]:
#get classifciation report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.84      0.84       312
           1       0.80      0.80      0.80       252

    accuracy                           0.82       564
   macro avg       0.82      0.82      0.82       564
weighted avg       0.82      0.82      0.82       564



[codecarbon INFO @ 18:33:22] Energy consumed for RAM : 0.000043 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:33:23] Energy consumed for all CPUs : 0.000027 kWh. Total CPU Power : 0.053500000000000006 W


In [81]:
svm_emissions = svmtracker.stop()
print(f'emissions from SVM train/testing run: {svm_emissions:5f} kg CO2 Equivalent')

[codecarbon INFO @ 18:33:24] Energy consumed for RAM : 0.000055 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:33:25] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.014857142857142859 W
[codecarbon INFO @ 18:33:25] 0.000082 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 18:33:26] Energy consumed for all CPUs : 0.000027 kWh. Total CPU Power : 0.046200000000000005 W
[codecarbon INFO @ 18:33:27] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0027000000000000006 W
[codecarbon INFO @ 18:33:27] 0.000082 kWh of electricity and 0.000000 L of water were used since the beginning.


emissions from SVM train/testing run: 0.000038 kg CO2 Equivalent


In [82]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred))

[[262  50]
 [ 51 201]]


**RECURRENT NEURAL NET (RNN)**

In [83]:
experiment_name = 'rnn'
rnntracker = EmissionsTracker(
    output_dir='codecarbon_mac/',
    output_file=f'{os_name}_{experiment_name}_emissions.csv')

[codecarbon WARNING @ 18:35:32] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 18:35:33] Error while trying to count physical CPUs: [Errno 2] No such file or directory: 'lscpu'. Defaulting to 1.
[codecarbon INFO @ 18:35:33] [setup] RAM Tracking...
[codecarbon INFO @ 18:35:33] [setup] CPU Tracking...
[codecarbon INFO @ 18:35:33] Tracking Apple CPU and GPU via PowerMetrics
[codecarbon INFO @ 18:35:33] [setup] GPU Tracking...
[codecarbon INFO @ 18:35:33] No GPU found.
[codecarbon INFO @ 18:35:33] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: PowerMetrics
                GPU Tracking Method: PowerMetrics
            
[codecarbon INFO @ 18:35:33] >>> Tracker's metadata:
[codecarbon INFO @ 18:35:33]   Platform system: macOS-15.6-arm64-arm-64bit
[codecarbon INFO @ 18:35:33]   Python version: 3.10.20
[codecarbon INFO @ 18:35:33]   CodeCarbon version: 

In [84]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


[]


In [85]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import metrics
from keras.layers import Dense,Input, GlobalMaxPooling1D, Dropout
from keras.layers import Conv1D, MaxPooling1D, Embedding, LSTM, SimpleRNN
from keras.models import Model, Sequential
from keras.initializers import Constant

In [86]:
MAX_NUM_WORDS = 20000
MAX_SEQUENCE_LENGTH = 300
VALIDATION_SPLIT = 0.2
EMBEDDING_DIM = 100 #going to use GLOVE if we end up doing embeddings

In [87]:
x_train, x_test, y_train, y_test=train_test_split(processed_descriptions,labels,test_size=0.3,random_state=42)

In [88]:
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS)
tokenizer.fit_on_texts(x_train)
train_sequences = tokenizer.texts_to_sequences(x_train)
test_sequences = tokenizer.texts_to_sequences(x_test)
word_index = tokenizer.word_index

In [89]:
trainvalid_data = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
test_data = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH)
trainvalid_labels = to_categorical(y_train, num_classes = 2)
test_labels = to_categorical(y_test, num_classes = 2) 

In [44]:
#think the below section should be moved to the top as we prob want to use GLOVE embeddings for everything

In [90]:
#time to split into train and validation
indices = np.arange(trainvalid_data.shape[0])
np.random.shuffle(indices)
trainvalid_data = trainvalid_data[indices]
trainvalid_labels = trainvalid_labels[indices]
num_validation_samples = int(0.3 * trainvalid_data.shape[0])
x_train = trainvalid_data[:-num_validation_samples]
y_train = trainvalid_labels[:-num_validation_samples]
x_val = trainvalid_data[-num_validation_samples:]
y_val = trainvalid_labels[-num_validation_samples:]

In [91]:
#making the embedding matrix
num_words = len(word_index) + 1
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
for word, i in word_index.items():
    if i > MAX_NUM_WORDS:
        continue
    embedding_vector = glove_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
    

In [92]:
embedding_layer = Embedding(num_words,
                            EMBEDDING_DIM,
                            embeddings_initializer = Constant(embedding_matrix),
                            input_length = MAX_SEQUENCE_LENGTH,
                            trainable=False)


In [106]:
rnnmodel = Sequential()
rnnmodel.add(embedding_layer)
rnnmodel.add(LSTM(256, dropout = 0.35))
rnnmodel.add(Dense(2, activation = 'softmax')) #len(labels_index)

rnnmodel.compile(loss='categorical_crossentropy',
                 optimizer = 'Adam',
                 metrics = ['acc', metrics.Precision(), metrics.Recall(), metrics.F1Score(average='macro')])

[codecarbon INFO @ 18:41:07] Energy consumed for all AppleSilicon GPUs : 0.000001 kWh. Total GPU Power : 0.0203 W
[codecarbon INFO @ 18:41:07] 0.000554 kWh of electricity and 0.000000 L of water were used since the beginning.


In [94]:
rnntracker.start()

In [107]:
#tf.debugging.set_log_device_placement(True)
rnn_train = rnnmodel.fit(x_train, y_train,
                         batch_size = 16,
                         epochs = 3,
                         validation_data = (x_val, y_val))

Epoch 1/3
27/58 ━━━━━━━━━━━━━━━━━━━━ 6s 203ms/step - acc: 0.5211 - f1_score: 0.5001 - loss: 0.7257 - precision_4: 0.5211 - recall_4: 0.5211

[codecarbon INFO @ 18:41:19] Energy consumed for RAM : 0.000196 kWh. RAM Power : 3.0 W


35/58 ━━━━━━━━━━━━━━━━━━━━ 4s 197ms/step - acc: 0.5221 - f1_score: 0.5051 - loss: 0.7226 - precision_4: 0.5221 - recall_4: 0.5221

[codecarbon INFO @ 18:41:20] Energy consumed for all CPUs : 0.000402 kWh. Total CPU Power : 10.123000000000001 W


43/58 ━━━━━━━━━━━━━━━━━━━━ 2s 191ms/step - acc: 0.5234 - f1_score: 0.5090 - loss: 0.7193 - precision_4: 0.5234 - recall_4: 0.5234

...

[codecarbon INFO @ 18:41:35] Energy consumed for all CPUs : 0.000437 kWh. Total CPU Power : 10.276299999999999 W


58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - acc: 0.6784 - f1_score: 0.6748 - loss: 0.6121 - precision_4: 0.6784 - recall_4: 0.6784

[codecarbon INFO @ 18:41:37] Energy consumed for all AppleSilicon GPUs : 0.000001 kWh. Total GPU Power : 0.0029 W
[codecarbon INFO @ 18:41:37] 0.000644 kWh of electricity and 0.000000 L of water were used since the beginning.


58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 200ms/step - acc: 0.6714 - f1_score: 0.6700 - loss: 0.6070 - precision_4: 0.6714 - recall_4: 0.6714 - val_acc: 0.6827 - val_f1_score: 0.6812 - val_loss: 0.5972 - val_precision_4: 0.6827 - val_recall_4: 0.6827
Epoch 3/3
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 197ms/step - acc: 0.7082 - f1_score: 0.7057 - loss: 0.5697 - precision_4: 0.7082 - recall_4: 0.7082 - val_acc: 0.7107 - val_f1_score: 0.7067 - val_loss: 0.5674 - val_precision_4: 0.7107 - val_recall_4: 0.7107


[codecarbon INFO @ 18:41:49] Energy consumed for RAM : 0.000216 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:41:50] Energy consumed for all CPUs : 0.000437 kWh. Total CPU Power : 0.023 W
[codecarbon INFO @ 18:41:52] Energy consumed for all AppleSilicon GPUs : 0.000001 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:41:52] 0.000654 kWh of electricity and 0.000000 L of water were used since the beginning.


In [ ]:
#recurrent_dropout > 0 majorly messed up performance and caused >1 min/step. 
#claude stated this parameter would cause it to fall back to CPU (and even though 
#i didnt see it in the messages, the insane increase in processing time when i removed
#this setting made me suspicious that it was silently falling back to cpu. 

In [108]:
loss, test_acc, test_precision, test_recall, test_f1 = rnnmodel.evaluate(test_data, test_labels)

18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - acc: 0.7216 - f1_score: 0.7204 - loss: 0.5610 - precision_4: 0.7216 - recall_4: 0.7216


[codecarbon INFO @ 18:42:04] Energy consumed for RAM : 0.000227 kWh. RAM Power : 3.0 W


In [109]:
rnn_emissions = rnntracker.stop()
print(f'Emissions from RNN train/test: {rnn_emissions:5f} kg CO2 eq')

[codecarbon INFO @ 18:42:04] Energy consumed for RAM : 0.000237 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:42:05] Energy consumed for all CPUs : 0.000437 kWh. Total CPU Power : 0.036300000000000006 W
[codecarbon INFO @ 18:42:06] Energy consumed for all CPUs : 0.000453 kWh. Total CPU Power : 4.6735 W
[codecarbon INFO @ 18:42:07] Energy consumed for all AppleSilicon GPUs : 0.000001 kWh. Total GPU Power : 0.0027000000000000006 W
[codecarbon INFO @ 18:42:07] 0.000691 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 18:42:07] Energy consumed for all AppleSilicon GPUs : 0.000001 kWh. Total GPU Power : 0.0183 W
[codecarbon INFO @ 18:42:07] 0.000691 kWh of electricity and 0.000000 L of water were used since the beginning.


Emissions from RNN train/test: 0.000318 kg CO2 eq


In [110]:
print(f'Test data accuracy {test_acc} \n',
      f'Test data precision {test_precision} \n',
      f'Test data Recall {test_recall} \n',
      f'Test data F1 Score {test_f1}')

Test data accuracy 0.7216312289237976 
 Test data precision 0.7216312289237976 
 Test data Recall 0.7216312289237976 
 Test data F1 Score 0.7204278707504272


**RANDOM FOREST**

In [111]:
experiment_name = 'randomforest'
rftracker = EmissionsTracker(
    output_dir='codecarbon_mac/',
    output_file=f'{os_name}_{experiment_name}_emissions.csv')

[codecarbon WARNING @ 18:44:13] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 18:44:13] Error while trying to count physical CPUs: [Errno 2] No such file or directory: 'lscpu'. Defaulting to 1.
[codecarbon INFO @ 18:44:13] [setup] RAM Tracking...
[codecarbon INFO @ 18:44:13] [setup] CPU Tracking...
[codecarbon INFO @ 18:44:14] Tracking Apple CPU and GPU via PowerMetrics
[codecarbon INFO @ 18:44:14] [setup] GPU Tracking...
[codecarbon INFO @ 18:44:14] No GPU found.
[codecarbon INFO @ 18:44:14] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: PowerMetrics
                GPU Tracking Method: PowerMetrics
            
[codecarbon INFO @ 18:44:14] >>> Tracker's metadata:
[codecarbon INFO @ 18:44:14]   Platform system: macOS-15.6-arm64-arm-64bit
[codecarbon INFO @ 18:44:14]   Python version: 3.10.20
[codecarbon INFO @ 18:44:14]   CodeCarbon version: 

In [112]:
from sklearn.ensemble import RandomForestClassifier

In [113]:
X_train, X_test, y_train, y_test = train_test_split(sentence_embedding_list, labels, 
                                                    test_size = 0.3,
                                                    random_state=1)

In [114]:
#parameters
n_estimators = [100, 500]
max_features = ['sqrt', 'log2']
max_depth = [None, 10, 25]

params = {
    'classify__n_estimators': n_estimators,
    'classify__max_features': max_features,
    'classify__max_depth': max_depth
}

In [115]:
pipe_RF = Pipeline([
    ('scale', StandardScaler()),
    ('classify',RandomForestClassifier())
])

In [116]:
inner_cv = KFold(n_splits = 3, shuffle = True, random_state = 1)
outer_cv = KFold(n_splits = 5, shuffle = True, random_state = 1)

grid_RF = GridSearchCV(pipe_RF, params, cv = inner_cv)

In [117]:
rftracker.start()

In [118]:
scores = cross_validate(grid_RF,
                        X=X_train,
                        y=y_train,
                        cv=outer_cv,
                        scoring = ['accuracy','precision','recall','f1'],
                       return_estimator = True)

[codecarbon INFO @ 18:44:48] Energy consumed for RAM : 0.000013 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:44:49] Energy consumed for all CPUs : 0.000023 kWh. Total CPU Power : 5.473 W
[codecarbon INFO @ 18:44:50] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:44:50] 0.000035 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 18:45:03] Energy consumed for RAM : 0.000023 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:45:04] Energy consumed for all CPUs : 0.000041 kWh. Total CPU Power : 5.2321 W
[codecarbon INFO @ 18:45:05] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:45:05] 0.000064 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 18:45:18] Energy consumed for RAM : 0.000033 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:45:19] Energy consumed for all CPUs : 0.000059 kWh. Total CPU Power : 

In [119]:
print(scores['test_accuracy'])
print(scores['test_precision'])
print(scores['test_recall'])
print(scores['test_f1'])

[0.76515152 0.83269962 0.84030418 0.81749049 0.80228137]
[0.81609195 0.83809524 0.90909091 0.79807692 0.82178218]
[0.60683761 0.76521739 0.73170732 0.75454545 0.70940171]
[0.69607843 0.8        0.81081081 0.77570093 0.76146789]


In [120]:
grid_RF.fit(X_train, y_train)
grid_RF.best_params_

[codecarbon INFO @ 18:46:33] Energy consumed for RAM : 0.000084 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:46:34] Energy consumed for all CPUs : 0.000131 kWh. Total CPU Power : 5.3772 W
[codecarbon INFO @ 18:46:35] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:46:35] 0.000215 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 18:46:35] 0.000806 g.CO2eq/s mean an estimation of 25.405126547093168 kg.CO2eq/year
[codecarbon INFO @ 18:46:48] Energy consumed for RAM : 0.000095 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:46:49] Energy consumed for all CPUs : 0.000149 kWh. Total CPU Power : 5.232 W
[codecarbon INFO @ 18:46:50] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0029000000000000002 W
[codecarbon INFO @ 18:46:50] 0.000243 kWh of electricity and 0.000000 L of water were used since the beginning.


{'classify__max_depth': None,
 'classify__max_features': 'log2',
 'classify__n_estimators': 500}

[codecarbon INFO @ 18:47:03] Energy consumed for RAM : 0.000105 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:47:04] Energy consumed for all CPUs : 0.000149 kWh. Total CPU Power : 0.027499999999999997 W
[codecarbon INFO @ 18:47:06] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0026999999999999997 W
[codecarbon INFO @ 18:47:06] 0.000254 kWh of electricity and 0.000000 L of water were used since the beginning.


In [121]:
rf = RandomForestClassifier(n_estimators=500, max_features='log2')
rf.fit(X_train, y_train)


,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'log2'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


[codecarbon INFO @ 18:47:18] Energy consumed for RAM : 0.000115 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:47:19] Energy consumed for all CPUs : 0.000149 kWh. Total CPU Power : 0.0231 W


In [122]:
y_pred = rf.predict(X_test)

[codecarbon INFO @ 18:47:21] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.011399999999999999 W
[codecarbon INFO @ 18:47:21] 0.000264 kWh of electricity and 0.000000 L of water were used since the beginning.


In [123]:
rf_emissions = rftracker.stop()
print(f'CO2 Equivalent emissions from Random Forest train/test: {rf_emissions:5f} kg')

[codecarbon INFO @ 18:47:25] Energy consumed for RAM : 0.000119 kWh. RAM Power : 3.0 W
[codecarbon INFO @ 18:47:26] Energy consumed for all CPUs : 0.000149 kWh. Total CPU Power : 0.022899999999999997 W
[codecarbon INFO @ 18:47:28] Energy consumed for all AppleSilicon GPUs : 0.000000 kWh. Total GPU Power : 0.0 W
[codecarbon INFO @ 18:47:28] 0.000268 kWh of electricity and 0.000000 L of water were used since the beginning.


CO2 Equivalent emissions from Random Forest train/test: 0.000123 kg


In [124]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.93      0.85       280
           1       0.92      0.74      0.82       284

    accuracy                           0.83       564
   macro avg       0.85      0.83      0.83       564
weighted avg       0.85      0.83      0.83       564



**BERT Transformer**

In [35]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from datasets import Dataset

In [38]:
model_df = model_df.reset_index(drop=True)
bert_data = Dataset.from_pandas(model_df)

In [68]:
bert_data = bert_data.rename_column('fraudulent','labels')

In [69]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [70]:
def preprocess_bert(text):
    return tokenizer(text['description'], truncation=True)

In [71]:
tokenized_desc = bert_data.map(preprocess_bert)

Map:   0%|          | 0/1880 [00:00<?, ? examples/s]

In [92]:
tokenized_desc = tokenized_desc.remove_columns(['description'])

In [83]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

In [93]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [94]:
id2label = {0:'REAL', 1:'FRAUDULENT'}
label2id = {'REAL':0, 'FRAUDULENT':1}

In [95]:
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",
                                                               num_labels=2,
                                                               id2label=id2label,
                                                               label2id=label2id)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [96]:
split_dataset = tokenized_desc.train_test_split(test_size=0.2,seed=42)
train_data = split_dataset['train']
test_data=split_dataset['test']

In [97]:
training_args = TrainingArguments(output_dir = 'first_bert_model',
                                  remove_unused_columns = False,
                                 push_to_hub = False)

trainer = Trainer(model=bert_model,
                 args=training_args,
                 train_dataset=train_data,
                 eval_dataset=test_data,
                 processing_class=tokenizer,
                 data_collator=data_collator) #placeholders

In [98]:
trainer.train()

Step,Training Loss


RuntimeError: MPS backend out of memory (MPS allocated: 10.02 GiB, other allocations: 8.03 GiB, max allowed: 18.13 GiB). Tried to allocate 96.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [91]:
train_data


Dataset({
    features: ['description', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1504
})